# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

/var/folders/9p/f66yhymx35939zzlg51g0rrc0000gn/T/ipykernel_14088/3307975217.py:5: DeprecationWarning: Please import from 'ax.generation_strategy.generation_strategy' instead of 'ax.modelbridge.generation_strategy'. The latter is deprecated and will be removed in a future release.
  from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


# generate recommendations

In [2]:
print("List of available drugs:")
print("=====================================")
for drug in hf.normalize_drug_properties_dict.keys():
    print(f"{drug}:   {hf.normalize_drug_properties_dict[drug]['full_name']}")

List of available drugs:
IBP:   Ibuprofen
DCF:   Diclofenac
LOV:   Lovastatin
ITZ:   Itraconazole
RPD:   Risperidone
GLV:   Griseofulvin
CTZ:   Clotrimazole
GBC:   Glibenclamide/Glyburide


In [3]:
drug = input("Enter the drug name (abbreviation): ")


print("Please confirm the following information:")
print()
print("Drug:                ", drug, " (",hf.normalize_drug_properties_dict[drug]['full_name'],")")
print("Stock solution conc: ", hf.normalize_drug_properties_dict[drug]['drug_stock_conc'], "mg/mL")
print("Molecular weight:    ",hf.normalize_drug_properties_dict[drug]['normalized_properties']['Drug_MW'] * 1000)
print("LogP:                ", hf.normalize_drug_properties_dict[drug]['normalized_properties']['Drug_LogP'] * 10)
print("TPSA:                ", hf.normalize_drug_properties_dict[drug]['normalized_properties']['Drug_TPSA'] * 1000)



print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:

Drug:                 GBC  ( Glibenclamide/Glyburide )
Stock solution conc:  25 mg/mL
Molecular weight:     494.0
LogP:                 3.6420000000000003
TPSA:                 113.60000000000001

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [4]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 26
Very Important: Please Confirm the Iteration Number is Iteration 26
Very Important: Please Confirm the Iteration Number is Iteration 26


In [5]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = drug, bopt = 1, n_trials=3)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

**************************************************************************************************************

Generating Bayesian Optimization trials for
Drug name:  Glibenclamide/Glyburide GBC  | Iteration:  26

**************************************************************************************************************


[INFO 07-10 10:29:59] ax.service.ax_client: Generated new trial 78 with parameters {'Drug_MW': 0.494, 'Drug_LogP': 0.3642, 'Drug_TPSA': 0.1136, 's1': 0, 's2': 28, 's3': 0, 's4': 0, 's5': 0, 's6': 0, 's7': 100, 's8': 0, 'surfactant_conc': 100, 'drug_conc': 100} using model SAASBO.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/core/data.py:293: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
[INFO 07-10 10:30:51] ax.service.ax_client: Generated new trial 79 with parameters {'Drug_MW': 0.494, 'Drug_LogP': 0.3642, 'Drug_TPSA': 0.1136, 's1': 0, 's2': 100, 's3': 0, 's4': 0, 's5': 0, 's6': 0, 's7': 100, 's8': 2, 'surfactant_conc': 100, 'drug_conc': 100} using model SAASBO.
/opt/a

Time taken for optimization: 3.33 mins
Time taken for optimization: 199.8 seconds


# process results

In [6]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 47, 's2': 60, 's3': 50, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 'surfactant_conc': 85, 'drug_conc': 100})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 74, 's2': 0, 's3': 88, 's4': 66, 's5': 22, 's6': 52, 's7': 58, 's8': 79, 'surfactant_conc': 35, 'drug_conc': 100})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 78, 's2': 88, 's3': 1, 's4': 3, 's5': 32, 's6': 94, 's7': 44, 's8': 58, 'surfactant_conc': 1, 'drug_conc': 100})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', paramete

In [7]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [8]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: H1
Deep plate will start at: E1

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [9]:
hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_26.py


In [10]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

,trial_index,success
0,0,0
1,1,0
2,2,0


In [11]:
results = hf.build_results(n, df_conc, df_absorbance)
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_input,drug_conc,success,complexity
0,78,0,28,0,0,0,0,100,0,50.0,25.0,0,2
1,79,0,100,0,0,0,0,100,2,50.0,25.0,0,3
2,80,0,0,0,0,0,0,84,15,50.0,25.0,0,2


In [12]:
norm_results = hf.normalize_data(results, 'normalize')

In [13]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_input,drug_conc,success,complexity
0,78,0,28,0,0,0,0,100,0,1.0,1.0,0.0,0.250
1,79,0,100,0,0,0,0,100,2,1.0,1.0,0.0,0.375
2,80,0,0,0,0,0,0,84,15,1.0,1.0,0.0,0.250


# load the results to the optimizer

In [14]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 07-10 11:16:25] ax.service.ax_client: Completed trial 78 with data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)}.
[INFO 07-10 11:16:25] ax.service.ax_client: Completed trial 79 with data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)}.
[INFO 07-10 11:16:25] ax.service.ax_client: Completed trial 80 with data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)}.


Trial 78: success=0 → overriding surfactant_input & complexity to 1
Trial 79: success=0 → overriding surfactant_input & complexity to 1
Trial 80: success=0 → overriding surfactant_input & complexity to 1


AxClient(experiment=Experiment(drug_surfactant))